# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates loading, overview, extraction, and exploration of the FAIR² dataset using the `mlcroissant` library, referencing dataset entities by their `@id`.

### Dataset Source
Dataset Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load dataset and print metadata name/description
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset loaded.")
print("Name:", metadata.name)
print("Description:", metadata.description)
print("Version:", metadata.version)
print("Identifier:", metadata.identifier)
# For completeness, print citeAs
print("Citation:", metadata.citeAs)

## 2. Data Overview
Review the available record sets and their IDs, and show fields/columns by `@id`.

In [ ]:
# List available record sets, fields, and columns by @id
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets available directly. Attempt to discover via metadata.")
else:
    print("Record Sets:")
    for rs in record_sets:
        print(f"- @id: {rs['@id']}")
        print(f"  name: {rs.get('name','')}")
        print(f"  description: {rs.get('description','')}")
        # List fields/columns
        fields = rs.get('field',[])
        if fields:
            print("  Fields:")
            for f in fields:
                print(f"    * {f['@id']} (name: {f.get('name', '')}, type: {f.get('dataType', '')})")
        columns = rs.get('column',[])
        if columns:
            print("  Columns:")
            for c in columns:
                print(f"    * {c['@id']} (name: {c.get('name','')}, type: {c.get('dataType','')})")
else:
    # If record_sets is empty, fetch record sets from Croissant schema
    # The metadata may have a 'recordSet' key listing IDs
    meta_dict = dataset.metadata.to_json()
    record_sets_ids = meta_dict.get('recordSet',[])
    print("Record Sets IDs from metadata:")
    for rsid in record_sets_ids:
        print(f"- @id: {rsid}")

## 3. Data Extraction
Load data from available record set(s) into pandas DataFrames for analysis. Use record set and field `@id`s from above.

In [ ]:
# Attempt to extract data from all record sets referenced in metadata
meta_dict = dataset.metadata.to_json()
record_sets_ids = meta_dict.get('recordSet',[])

dataframes = {}
for rsid in record_sets_ids:
    try:
        records = list(dataset.records(record_set=rsid))
        if records:
            df = pd.DataFrame(records)
            dataframes[rsid] = df
            print(f"Loaded DataFrame for Record Set {rsid}")
            print(f"Columns: {df.columns.tolist()}")
            print(f"First records:")
            print(df.head())
        else:
            print(f"No records found for Record Set {rsid}")
    except Exception as e:
        print(f"Could not load records for Record Set {rsid}: {e}")

# Select first available record set for downstream EDA
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Main record set for EDA: {main_record_set_id}")
    print("Columns available:", dataframes[main_record_set_id].columns.tolist())
else:
    main_record_set_id = None
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common filtering, normalization, and grouping using fields by their `@id`s. Try to use one numeric field and one grouping/categorical field for demonstration.

In [ ]:
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Find a numeric field to use for demonstration
    numeric_fields = [col for col in df.columns if df[col].dtype in ['int64','float64'] or df[col].apply(lambda x: isinstance(x,(int,float))).all()]
    print("Numeric fields candidates:", numeric_fields)
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
    else:
        numeric_field_id = None

    # Try threshold = 10, but auto fallback if values are small
    if numeric_field_id:
        threshold = 10
        try:
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold}:")
            print(filtered_df.head())
            # Normalization
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id,f"{numeric_field_id}_normalized" ]].head())
        except Exception as e:
            print("Could not filter/normalize numeric field:", e)
        
        # Find group field (categorical)
        group_candidates = [col for col in df.columns if df[col].dtype=='object' or df[col].apply(lambda x: isinstance(x,str)).all()]
        if group_candidates:
            group_field_id = group_candidates[0]
            try:
                grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
                print(f"Grouped by {group_field_id} (mean):")
                print(grouped_df.head())
            except Exception as e:
                print("Could not group by field:", e)
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found in DataFrame.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field and the relationship between numeric and categorical fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Use previous selection
    if 'numeric_field_id' in locals() and numeric_field_id:
        # Histogram
        plt.figure(figsize=(6,4))
        df[numeric_field_id].dropna().hist()
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()
        
        # If a grouping field is available, show boxplot
        if 'group_field_id' in locals() and group_field_id:
            plt.figure(figsize=(10,5))
            sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.xticks(rotation=45)
            plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
This notebook demonstrates stepwise exploration of the Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer dataset using the `mlcroissant` library:
- Loaded dataset and reviewed metadata using the Croissant schema.
- Listed available record sets and fields by their `@id`.
- Extracted records for each record set and loaded into pandas DataFrames.
- Performed filtering, normalization, and grouping by field `@id`s.
- Visualized numeric distributions and relations to categorical fields.

For further analysis, reference all fields/entities with their `@id` from the Croissant schema for reproducible FAIR workflows.